In [1]:
import torch

print("CUDA Status:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Model:", torch.cuda.get_device_name(0))

c:\Users\dloc\miniconda3\envs\carla_rl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA Status: True
GPU Model: NVIDIA GeForce GTX 1650 with Max-Q Design


In [27]:

import carla
# Kết nối tới server CARLA
SERVER_HOST = 'localhost'  # Đổi thành IP server nếu notebook chạy ở máy khác
SERVER_PORT = 2000

client = carla.Client(SERVER_HOST, SERVER_PORT)
client.set_timeout(30.0)
world = client.get_world()
print('Đã kết nối CARLA tại {}:{} | Map: {}'.format(
    SERVER_HOST, SERVER_PORT, world.get_map().name
))

Đã kết nối CARLA tại localhost:2000 | Map: Town03


In [33]:
for tl in world.get_actors().filter('traffic.traffic_light*'):
    tl.freeze(False)
    tl.set_green_time(8.0)
    tl.set_yellow_time(2.0)
    tl.set_red_time(3.0)

In [3]:
# Xem danh sách các bản đồ có sẵn
print(client.get_available_maps())


['/Game/Carla/Maps/Town01', '/Game/Carla/Maps/Town02', '/Game/Carla/Maps/Town03', '/Game/Carla/Maps/Town04', '/Game/Carla/Maps/Town05']


In [19]:
# Lệnh chuyển map01
client.load_world('Town01')

In [22]:
# Lệnh chuyển map02
client.load_world('Town02')

In [31]:
# Lệnh chuyển map03
client.load_world('Town03')

In [15]:
# Lệnh chuyển map04
client.load_world('Town04')

In [28]:
# Lệnh chuyển map05
client.load_world('Town05')

# Điều khiển thời tiết CARLA
Chạy cell tạo hàm một lần, sau đó chạy cell preset hoặc thời tiết tùy chỉnh. Các hàm luôn lấy world hiện tại nên vẫn dùng được sau khi đổi map.

In [4]:
# Tự lấy toàn bộ preset từ đúng phiên bản CARLA đang chạy
WEATHER_PRESETS = {
    name: getattr(carla.WeatherParameters, name)
    for name in dir(carla.WeatherParameters)
    if name and name[0].isupper()
}

def set_weather_preset(name):
    if name not in WEATHER_PRESETS:
        available = ', '.join(sorted(WEATHER_PRESETS))
        raise ValueError('Preset không hợp lệ. Các preset hiện có: ' + available)
    world = client.get_world()
    world.set_weather(WEATHER_PRESETS[name])
    print('Đã đặt thời tiết:', name, '| Map:', world.get_map().name)
    return world.get_weather()

print('Preset thời tiết có thể dùng:')
print(', '.join(sorted(WEATHER_PRESETS)))

Preset thời tiết có thể dùng:
ClearNoon, ClearSunset, CloudyNoon, CloudySunset, Default, HardRainNoon, HardRainSunset, MidRainSunset, MidRainyNoon, SoftRainNoon, SoftRainSunset, WetCloudyNoon, WetCloudySunset, WetNoon, WetSunset


## 1. Nhóm trời quang đãng

In [19]:
world.set_weather(carla.WeatherParameters.Default)          # Thời tiết mặc định (trời quang, nắng trưa)

In [16]:
world.set_weather(carla.WeatherParameters.ClearNoon)         # Giữa trưa quang đãng, nắng gắt, bóng ngắn, đường khô

In [31]:
world.set_weather(carla.WeatherParameters.ClearSunset)       # Hoàng hôn quang đãng, nắng cam chiếu góc thấp, bóng dài

In [5]:
world.set_weather(carla.WeatherParameters.CloudyNoon)        # Giữa trưa nhiều mây, ánh sáng đều khuếch tán, đường khô

In [29]:
world.set_weather(carla.WeatherParameters.CloudySunset)      # Hoàng hôn nhiều mây, khung cảnh u tối

## 2. Nhóm mặt đường ướt không có mưa

In [21]:
world.set_weather(carla.WeatherParameters.WetNoon)           # Giữa trưa đường ướt, bóng nước phản chiếu ánh nắng trưa mạnh

In [7]:
world.set_weather(carla.WeatherParameters.WetSunset)         # Hoàng hôn đường ướt, phản chiếu ánh sáng cam tà

In [32]:
world.set_weather(carla.WeatherParameters.WetCloudyNoon)     # Giữa trưa nhiều mây, đường bóng ướt, không mưa rơi

In [22]:
world.set_weather(carla.WeatherParameters.WetCloudySunset)   # Hoàng hôn nhiều mây, đường ướt, thiếu sáng và phản chiếu lớn

## 3. Nhóm có mưa rơi từ nhẹ -> lớn

In [18]:
world.set_weather(carla.WeatherParameters.SoftRainNoon)      # Giữa trưa mưa nhỏ, hạt mưa thưa, đường ẩm nhẹ

In [11]:
world.set_weather(carla.WeatherParameters.SoftRainSunset)    # Hoàng hôn mưa nhỏ, ánh sáng nhập nhạng

In [30]:
world.set_weather(carla.WeatherParameters.MidRainyNoon)      # Giữa trưa mưa vừa, có giọt nước đọng kính/camera

In [24]:
world.set_weather(carla.WeatherParameters.MidRainSunset)     # Hoàng hôn mưa vừa, tầm nhìn bắt đầu suy giảm

In [34]:
world.set_weather(carla.WeatherParameters.HardRainNoon)      # Giữa trưa mưa to, hạt mưa dày đặc, đường đọng nhiều nước

In [35]:
world.set_weather(carla.WeatherParameters.HardRainSunset)    # Hoàng hôn mưa to, tối, đường ướt đọng nước, tầm nhìn rất kém